In [1]:
import os
import sys
import yaml
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from geopy.distance import geodesic
# from ydata_profiling import ProfileReport
import sqlite3
import mlflow
import mlflow.sklearn
import mlflow.data
import pandas as pd
from sklearn.model_selection import train_test_split
from category_encoders import TargetEncoder
# from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold
from imblearn.over_sampling import SMOTE
import category_encoders as ce
from sklearn.preprocessing import OrdinalEncoder
sns.set(style="whitegrid")
import joblib
import xgboost as xgb


In [5]:
path = os.path.abspath(os.path.join(os.getcwd(),".."))
sys.path.append(path)

from src.utils import *
from src.constants import *

### Feature Pipeline

In [3]:
def split_data(df,VAL_SPLIT_RATIO):
    X = df.drop(TARGET, axis=1)
    y = df[TARGET]
    
    # Separate training and validation sets from the remaining data (e.g., 25% of X_train_val for validation)
    # This will result in train: 60%, validation: 20%, test: 20% of the original data
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=VAL_SPLIT_RATIO, random_state=42, stratify=y
    )

    train_df = pd.concat([X_train, y_train], axis=1)
    validation_df = pd.concat([X_val, y_val], axis=1)
    
    return X_train, X_val, y_train, y_val, train_df, validation_df


def feature_transform(df):
    # Create Date time features
    df[TRANS_DATE_TRANS_TIME] = pd.to_datetime(df[TRANS_DATE_TRANS_TIME])
    df[HOUR] = df[TRANS_DATE_TRANS_TIME].dt.hour
    df[DAY] = df[TRANS_DATE_TRANS_TIME].dt.dayofweek

    df[DOB] = pd.to_datetime(df[DOB])
    df[AGE] = (pd.to_datetime('today') - df[DOB]).dt.days // 365
    # Distance between user and merchant
    df[DISTANCE] = df.apply(lambda row: geodesic((row[LAT], row[LONG]),(row[MERCH_LAT], row[MERCH_LONG])).km, axis=1)
    # Log-transform amount to reduce skew
    df[LOG_AMT] = np.log1p(df[AMT])
 
    # Map to new category bins
    df[CATEGORY_BINNED] = df[CATEGORY].map(CATEGORY_MAP)
    
    # Replace any NaNs with 'Other' (fallback category)
    df[CATEGORY_BINNED] = df[CATEGORY_BINNED].fillna('Other')
    
    # Convert to categorical dtype
    df[CATEGORY_BINNED] = pd.Categorical(df[CATEGORY_BINNED], categories=['household', 'personal', 'other'])
    
    return df


def oh_encoding(df):
    #### 1. Category Binning – OneHotEncoder
    # i. Initialize encoder (fit on training data only)
    encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')  # drop='first' avoids multicollinearity
    # ii. Fit and transform training data
    encoded_array = encoder.fit_transform(df[[CATEGORY_BINNED]])
    # iii. Create DataFrame from encoded array with proper column names and index
    encoded_df = pd.DataFrame(encoded_array,
                              columns=encoder.get_feature_names_out([CATEGORY_BINNED]),
                              index=df.index)
    
    # iv. Concatenate encoded columns with original dataframe
    # Optionally drop or keep original 'category_binned' column — usually drop to avoid duplication
    df = pd.concat([df.drop(columns=[CATEGORY_BINNED]), encoded_df], axis=1)
    # Save OneHotEncoder
    joblib.dump(encoder, os.path.join(ARTIFACT_PATH, "onehot_encoder.pkl"))
    print('Onehot encoding of Category column is complete')
    return df, encoded_df
    

def target_encoding(X,y):
    #### 3. Job Target Encoding
    target_enc = ce.TargetEncoder(cols=[JOB], smoothing=0.3)
    target_enc.fit(X, y)
    X[JOB_TE] = target_enc.transform(X)[JOB]
    
    # Save TargetEncoder
    joblib.dump(target_enc, os.path.join(ARTIFACT_PATH, "target_encoder_job.pkl"))
    print('Target encoding of Job column is complete')
    return X

    
def ordinal_encoding(X):
    #### 4. State Ordinal Encoding
    ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[STATES_ENC] = ord_enc.fit_transform(X[[STATE]])

    joblib.dump(ord_enc, os.path.join(ARTIFACT_PATH, "ordinal_encoder_state.pkl"))
    print('Ordinal encoding of states column is complete')
    return X


def apply_smote(X, y):
    smote = SMOTE(sampling_strategy='auto', random_state=42)
    X_res, y_res = smote.fit_resample(X, y)
    return X_res, y_res


def process_train_set(X_train, y_train):
    ### TRAINING SET ###
    # Feature transformation on training data
    X_train = feature_transform(X_train)

    # Encodings on the training data
    ## Logging into mlflow
    with mlflow.start_run(run_name=FEATURE_RUN_NAME):
        #### 1. Category Binning – OneHotEncoder
        X_train, encoded_df = oh_encoding(X_train)
        # Logging to mlflow
        onehot_encoder_path = os.path.join(ARTIFACT_PATH, "onehot_encoder.pkl")
        mlflow.log_artifact(onehot_encoder_path, artifact_path="encoders")
    
        #### 2. Gender Encoding
        X_train[GENDER] = X_train[GENDER].map({'M': 1, 'F': 0})
        X_train[GENDER] = X_train[GENDER].fillna(0)
        
        #### 3. Job Target Encoding
        X_train = target_encoding(X_train, y_train)
        # Logging to mlflow
        target_encoder_path = os.path.join(ARTIFACT_PATH, "target_encoder_job.pkl")
        mlflow.log_artifact(target_encoder_path, artifact_path="encoders")
    
        #### 4. State Ordinal Encoding
        X_train = ordinal_encoding(X_train)
        # Logging to mlflow
        ordinal_encoder_path = os.path.join(ARTIFACT_PATH, "ordinal_encoder_state.pkl")
        mlflow.log_artifact(ordinal_encoder_path, artifact_path="encoders")

        # Log info about encodings
        mlflow.log_param("onehot_columns", list(encoded_df.columns))
        mlflow.log_param("target_encoded_column", "job")
        mlflow.log_param("ordinal_encoded_column", "state")
        mlflow.set_tag("stage", FEATURE_RUN_NAME)

    X_train = X_train.drop(columns=DROP_COLS)
    
    X_train, y_train = apply_smote(X_train, y_train)

    return X_train, y_train


def process_val_set(X_val, y_val):  
    # Feature transformation on Validation data
    X_val = feature_transform(X_val)
    
    # get latest Mlflow run_id
    run_id = get_mlflow_run_id(MLFLOW_EXP_NAME, FEATURE_RUN_NAME)
    # Load the latest encoders from Mlflow artifacts
    onehot_encoder, target_encoder, ordinal_encoder = load_encoders(run_id)
    
    ##### 1. ENCODING "category_binned" COLUMN
    # Encode and turn into DataFrame
    encoded_val = onehot_encoder.transform(X_val[['category_binned']])
    encoded_val_df = pd.DataFrame(encoded_val,
                                  columns=onehot_encoder.get_feature_names_out(['category_binned']),
                                  index=X_val.index)
    
    X_val = pd.concat([X_val.drop(columns=['category_binned']), encoded_val_df], axis=1)
    
    ##### 2. ENCODING "gender" COLUMN
    X_val['gender'] = X_val['gender'].map({'M': 1, 'F': 0})
    
    ##### 3. ENCODING "job" COLUMN
    X_val["job_te"] = target_encoder.transform(X_val)["job"]
    
    ##### 4. ENCODING "state" COLUMN
    X_val['states_enc'] = ordinal_encoder.transform(X_val[['state']])
    
    X_val["gender"] = X_val["gender"].fillna(0)
    
    X_val = X_val.drop(columns=DROP_COLS)

    return X_val, y_val


def get_mlflow_run_id(MLFLOW_EXP_NAME, RUN_NAME):
    # Set experiment
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXP_NAME)
    experiment_id = experiment.experiment_id
    
    # '{'run_name', 'user_id', 'status', 'end_time', 'Created', 'start_time', 'run_id', 'artifact_uri', 'created'}'
    
    # Search for the latest run where you logged encoding artifacts (e.g., by tag)
    runs = mlflow.search_runs(
        experiment_ids=[experiment_id],
        filter_string=f'tags.stage = "{RUN_NAME}"',
        order_by=["start_time DESC"]
    )
    
    if not runs.empty:
        # Get the MLflow run id of the most recent relevant run
        run_id = runs.iloc[0]["run_id"]
        print(f"Using run_id: {run_id}")
    else:
        raise ValueError("No relevant MLflow run found.")

    return run_id


def load_encoders(run_id):
    onehot_encoder = joblib.load(
        mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="encoders/onehot_encoder.pkl")
    )
    target_encoder = joblib.load(
        mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="encoders/target_encoder_job.pkl")
    )
    ordinal_encoder = joblib.load(
        mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="encoders/ordinal_encoder_state.pkl")
    )

    return onehot_encoder, target_encoder, ordinal_encoder

In [4]:
def feature_engg():
    # Set MLflow tracking URI (replace the path accordingly)_n
    mlflow.set_tracking_uri(TRACKING_URI)
    
    mlflow.set_experiment(MLFLOW_EXP_NAME)
    
    # load training and validation data from the DB
    df = load_from_sqlite(DB_PATH, TRAIN_VAL_TABLE)
    
    # Spliting the data into training and validation set
    X_train, X_val, y_train, y_val, train_df, validation_df = split_data(df, VAL_SPLIT_RATIO)
    
    X_train, y_train = process_train_set(X_train, y_train)
    
    # Save the training datasets into the DB
    train_df_final = pd.concat([X_train,y_train], axis=1)
    save_to_sqlite(train_df_final, DB_PATH, TRAINING_TABLE)
    
    ### VALIDATION SET ###
    X_val, y_val = process_val_set(X_val, y_val)
    
    val_df_final = pd.concat([X_val,y_val], axis=1)
    
    # Save the validation datasets into the DB
    save_to_sqlite(val_df_final, DB_PATH, VALIDATION_TABLE)

    print("feature transformations on the training and validation dataset is complete!")


# if __name__ == "__main__":
#     feature_engg()

In [5]:
feature_engg()

train_val_data loaded as df successfully!
Onehot encoding of Category column is complete
Target encoding of Job column is complete
Ordinal encoding of states column is complete
processed_train_data saved to the database


/home/vineet79ankam/miniconda3/envs/mlops-kubeflow/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using run_id: 83bb203874ed4c7e891a195157f39aa3


processed_validation_data saved to the database
feature transformations on the training and validation dataset is complete!


### Training Pipeline

In [6]:
# steps/train.py

import sqlite3
import pandas as pd
import pickle
import os
import mlflow
import mlflow.sklearn
import xgboost as xgb
import optuna
from sklearn.metrics import classification_report, roc_auc_score
import functools

def objective(trial, X_train, X_val, y_train, y_val):
    trial_number = trial.number
    
    MODEL_PARAMS = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "use_label_encoder": False,
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_loguniform("reg_alpha", 1e-8, 1.0),
        "reg_lambda": trial.suggest_loguniform("reg_lambda", 1e-8, 1.0),
        "scale_pos_weight": (len(y_train) - sum(y_train)) / sum(y_train)
    }
    
    clf = xgb.XGBClassifier(
        **MODEL_PARAMS,
        early_stopping_rounds=10,
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    preds = clf.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)

    # Log trial as a child run with a clear name
    with mlflow.start_run(run_name=f"optuna_trial_{trial_number}", nested=True):
        mlflow.log_params(MODEL_PARAMS)
        mlflow.log_metric("val_auc", auc)

    return auc


def train_model():

    mlflow.set_experiment(MLFLOW_EXP_NAME)

    with mlflow.start_run(run_name=HYPT_RUN_NAME) as parent_run:
        # Load data
        # load training data
        train_df = load_from_sqlite(DB_PATH, TRAINING_TABLE)
        X_train = train_df.drop(columns=[TARGET])
        y_train = train_df[TARGET]
        
        # Load validation data
        val_df = load_from_sqlite(DB_PATH, VALIDATION_TABLE)
        X_val = val_df.drop(columns=[TARGET])
        y_val = val_df[TARGET]

        # Wrap your objective so X_train, X_val, y_train, y_val are passed automatically
        objective_with_data = functools.partial(
            objective,
            X_train=X_train,
            X_val=X_val,
            y_train=y_train,
            y_val=y_val
        )
        
        # # Hyperparameter tuning
        # study = optuna.create_study(direction="maximize")
        # study.optimize(objective, X_train, X_val, y_train, y_val, n_trials=15, show_progress_bar=True)

        # Run Optuna study
        study = optuna.create_study(direction="maximize")
        study.optimize(objective_with_data, n_trials=15, show_progress_bar=True)

        
        best_params = study.best_trial.params
        best_auc = study.best_trial.value

        # Log hyperparameter tuning results
        mlflow.log_params(best_params)
        mlflow.log_metric("best_val_auc", best_auc)
        mlflow.log_param("n_trials", 15)
        mlflow.set_tag("stage", "hyperparameter_tuning")

        # Final training run as a nested run
        with mlflow.start_run(run_name=TRAINING_RUN_NAME, nested=True):

            FINAL_MODEL_PARAMS = {
                "objective": "binary:logistic",
                "eval_metric": "auc",
                "use_label_encoder": False,
                "scale_pos_weight": (len(y_train) - sum(y_train)) / sum(y_train),
                "random_state": 42,
                "n_jobs": -1,
                "early_stopping_rounds": 10
            }
            
            final_params = {**best_params, **FINAL_MODEL_PARAMS}
            
            model = xgb.XGBClassifier(**final_params)
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

            preds = model.predict(X_val)
            proba = model.predict_proba(X_val)[:, 1]

            auc = roc_auc_score(y_val, proba)
            report = classification_report(y_val, preds, output_dict=True)

             # Log final model parameters and metrics
            mlflow.log_params(final_params)
            mlflow.log_metric("final_val_auc", auc)
            mlflow.log_metrics({f"final_val_{k}": v for k, v in report["1"].items()})

            # Additional metrics from classification report
            mlflow.log_metric("final_val_accuracy", report["accuracy"])
            mlflow.log_metric("final_val_macro_f1", report["macro avg"]["f1-score"])
            mlflow.log_metric("final_val_weighted_f1", report["weighted avg"]["f1-score"])

            # Create artifacts directory if it doesn't exist
            os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

            # Use MLflow's built-in XGBoost model logging (RECOMMENDED)
            try:
                mlflow.xgboost.log_model(
                    xgb_model=model,
                    artifact_path="xgb_model",
                    registered_model_name="fd_xgb_model"  # Optional: registers in model registry
                )
                print("✅ Model logged using mlflow.xgboost.log_model")
            except Exception as e:
                print(f"❌ Error with xgboost.log_model: {e}")
                # Fallback to sklearn logging
                mlflow.sklearn.log_model(
                    sk_model=model,
                    artifact_path="xgb_model",
                    registered_model_name="fd_xgb_model"
                )
                print("✅ Model logged using mlflow.sklearn.log_model (fallback)")

        print("Training complete. Best AUC:", auc)


# Optional: Add this function to load the model later
def load_model_from_mlflow(run_id, model_artifact_path="xgb_model"):
    """Load model from MLflow run"""
    model_uri = f"runs:/{run_id}/{model_artifact_path}"
    try:
        model = mlflow.xgboost.load_model(model_uri)
        print(f"✅ Model loaded from MLflow: {model_uri}")
        return model
    except:
        # Fallback to sklearn loading
        model = mlflow.sklearn.load_model(model_uri)
        print(f"✅ Model loaded from MLflow (sklearn): {model_uri}")
        return model
        
# if __name__ == "__main__":
#     train_model()

In [7]:
train_model()

processed_train_data loaded as df successfully!


[I 2025-08-21 18:42:00,860] A new study created in memory with name: no-name-14b20417-030f-4a79-b590-b7715ac243b2


processed_validation_data loaded as df successfully!


  0%|                                                                                                          | 0/15 [00:00<?, ?it/s]/tmp/ipykernel_4655/4091389713.py:28: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "reg_alpha": trial.suggest_loguniform("reg_alpha", 1e-8, 1.0),
/tmp/ipykernel_4655/4091389713.py:29: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "reg_lambda": trial.suggest_loguniform("reg_lambda", 1e-8, 1.0),
/home/vineet79ankam/miniconda3/envs/mlops-kubeflow/lib/python3.11/site-packages/xgboost/callback.py:386: UserWarning: [18:42:03] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = mod

[I 2025-08-21 18:42:07,118] Trial 0 finished with value: 0.9652547802048362 and parameters: {'n_estimators': 205, 'max_depth': 3, 'learning_rate': 0.1196878227078146, 'subsample': 0.8738967130217125, 'colsample_bytree': 0.8743324709251549, 'min_child_weight': 8, 'gamma': 2.3988562182522384, 'reg_alpha': 0.04767833035432357, 'reg_lambda': 0.0001742312117631336}. Best is trial 0 with value: 0.9652547802048362.


...

[I 2025-08-21 18:43:40,109] Trial 13 finished with value: 0.9754076207986507 and parameters: {'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.010604941537456624, 'subsample': 0.8208940757188224, 'colsample_bytree': 0.9503128075083351, 'min_child_weight': 5, 'gamma': 4.953095098564663, 'reg_alpha': 0.005259888636546403, 'reg_lambda': 1.0467868180240142e-08}. Best is trial 11 with value: 0.9786735927583504.


/home/vineet79ankam/miniconda3/envs/mlops-kubeflow/lib/python3.11/site-packages/xgboost/callback.py:386: UserWarning: [18:43:40] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
Best trial: 11. Best value: 0.978674: 100%|███████████████████████████████████████████████████████████| 15/15 [01:46<00:00,  7.09s/it]


[I 2025-08-21 18:43:47,187] Trial 14 finished with value: 0.9634531972983335 and parameters: {'n_estimators': 399, 'max_depth': 7, 'learning_rate': 0.010037780505929357, 'subsample': 0.9980995159945468, 'colsample_bytree': 0.9966057995339007, 'min_child_weight': 4, 'gamma': 3.474508661800519, 'reg_alpha': 0.6823635655624269, 'reg_lambda': 3.8670842347444417e-07}. Best is trial 11 with value: 0.9786735927583504.


/home/vineet79ankam/miniconda3/envs/mlops-kubeflow/lib/python3.11/site-packages/xgboost/callback.py:386: UserWarning: [18:43:48] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
2025/08/21 18:43:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/home/vineet79ankam/miniconda3/envs/mlops-kubeflow/lib/python3.11/site-packages/xgboost/sklearn.py:1028: UserWarning: [18:43:56] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2025/08/21 18:44:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'fd_xgb_model'.
Created version '1' of model 'fd_x

✅ Model logged using mlflow.xgboost.log_model
Training complete. Best AUC: 0.9786735927583504


In [ ]:
# steps/inference.py
"""
Load latest MLflow model from experiment and run inference on test set from SQLite.
Saves predictions to SQLite table and logs inference metrics to MLflow.
"""

import os
import mlflow
import mlflow.pyfunc
import pandas as pd
import sqlite3
from mlflow.tracking import MlflowClient
from sklearn.metrics import roc_auc_score, classification_report
from pathlib import Path

DB_PATH = "db/fraud_data.db"
MLFLOW_EXPERIMENT = "FraudDetectionTraining"
PREDICTIONS_TABLE = "predictions"
ARTIFACT_DIR = "artifacts"

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "file:./mlruns"))


def load_latest_model_uri(experiment_name: str):
    client = MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)
    if exp is None:
        raise RuntimeError(f"MLflow experiment not found: {experiment_name}")

    # Search runs sorted by start_time desc
    runs = client.search_runs(exp.experiment_id, order_by=["attribute.start_time DESC"], max_results=50)
    # find the latest run that logged a model artifact under "model"
    for run in runs:
        run_id = run.info.run_id
        # Check if artifacts contain 'model' - this may vary depending on how the model was logged
        # We attempt to construct a runs:/<run_id>/model URI and let MLflow handle missing
        candidate = f"runs:/{run_id}/model"
        try:
            # quick check: load model; if it errors, continue to next run
            _ = mlflow.pyfunc.load_model(candidate)
            return candidate, run_id
        except Exception:
            continue
    raise RuntimeError("No valid MLflow model found in experiment runs")


def load_test_df_from_sqlite(db_path):
    conn = sqlite3.connect(db_path)
    df = pd.read_sql("SELECT * FROM test", conn)
    conn.close()
    return df


def save_predictions_to_sqlite(df_preds, db_path, table_name=PREDICTIONS_TABLE):
    conn = sqlite3.connect(db_path)
    df_preds.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()


def run_inference():
    model_uri, run_id = load_latest_model_uri(MLFLOW_EXPERIMENT)
    print("Loading model from:", model_uri)
    model = mlflow.pyfunc.load_model(model_uri)

    df_test = load_test_df_from_sqlite(DB_PATH)
    X_test = df_test.drop(columns=["is_fraud"])
    y_test = df_test["is_fraud"]

    preds_proba = model.predict(X_test)  # for pyfunc, returns probability if model returns it
    # If model returns single class probabilities or arrays, adapt accordingly:
    # If predict returns array of probabilities, ensure shape (n,)
    if isinstance(preds_proba, (list,)):
        preds_proba = pd.Series(preds_proba)
    elif isinstance(preds_proba, (pd.DataFrame, pd.Series)):
        # if model returned dataframe (e.g., two columns 0/1 probs), get column for class 1
        if isinstance(preds_proba, pd.DataFrame) and 1 in preds_proba.columns:
            preds_proba = preds_proba[1]
        else:
            # try first column
            preds_proba = preds_proba.iloc[:, 0] if preds_proba.shape[1] == 1 else preds_proba.iloc[:, -1]

    preds_label = (preds_proba >= 0.5).astype(int)

    # Metrics
    auc = roc_auc_score(y_test, preds_proba)
    report = classification_report(y_test, preds_label, output_dict=True)

    # Save predictions and inputs
    out = df_test.copy()
    out["pred_proba"] = preds_proba
    out["pred_label"] = preds_label

    save_predictions_to_sqlite(out, DB_PATH)

    # Log to MLflow under a dedicated inference experiment (optional)
    mlflow.set_experiment("FraudDetectionInference")
    with mlflow.start_run(run_name=f"inference_run_from_{run_id}"):
        mlflow.log_metric("test_auc", float(auc))
        mlflow.log_dict(report, "inference_classification_report.json")
        # optionally log small sample of predictions
        sample_csv = os.path.join(ARTIFACT_DIR, "inference_sample.csv")
        Path(ARTIFACT_DIR).mkdir(parents=True, exist_ok=True)
        out.head(200).to_csv(sample_csv, index=False)
        mlflow.log_artifact(sample_csv)

    print("Inference metrics logged; predictions written to DB table:", PREDICTIONS_TABLE)


if __name__ == "__main__":
    run_inference()


In [ ]:
# project_root = os.path.abspath(os.path.join(os.getcwd(),".."))
# Setup the file path
file_path = os.path.join(project_root, "data", "inference_data.csv")

inf_df = pd.read_csv(file_path)

# Separate features (X) and target (y)
# Replace 'target_column' with the actual name of your target column (e.g., 'is_fraud')
X_test = inf_df.drop('is_fraud', axis=1)
y_test = inf_df['is_fraud']


In [ ]:
# Create Date time features
X_test['trans_date_trans_time'] = pd.to_datetime(X_test['trans_date_trans_time'])
X_test['dob'] = pd.to_datetime(X_test['dob'])

# Create new features
X_test['hour'] = X_test['trans_date_trans_time'].dt.hour
X_test['day'] = X_test['trans_date_trans_time'].dt.dayofweek
X_test['age'] = (pd.to_datetime('today') - X_test['dob']).dt.days // 365


# Distance between user and merchant
X_test['distance'] = X_test.apply(lambda row: geodesic((row['lat'], row['long']),(row['merch_lat'],row['merch_long'])).km, axis=1)


# Log-transform amount to reduce skew
X_test['log_amt'] = np.log1p(X_test['amt'])